# NB02 — Rotate the cached activation dataset

The transform may be done on a CPU. 
The live-vs-offline spot check loads Qwen3-8B so it needs a GPU.

`M_Q` is functionally identical to `M`, so every `has_changed` label and every content string is unchanged and the counterfact patching
pipeline does not need rerunning. 
Load the cached tensors, apply `v ↦ Qv`, write out as a sibling dataset.

This notebook creates the three datasets for the sweep notebooks.
We build them from a single shuffled base dataset, so the data ordering is the same and the train/eval split are the same for every arm.
The only thing that makes a difference is the vectors.


In [ ]:
# --- dependencies -----------------------------------------------------------
# A RunPod image ships torch built against the pod's own driver, so nothing here may
# replace it: se_env pins the installed torch as a pip constraint and installs only what
# is missing or below the floor these notebooks need — including bitsandbytes, which Colab
# had preinstalled and RunPod images do not (se_common asks for paged_adamw_8bit).
# Safe to re-run: a no-op on a warm pod.
import os
import sys

# `se/` holds the shared modules: se_env, se_config, rotate, se_common. Clone this repo
# onto the pod's volume (/workspace/self_explainer) so it survives the pod.
REPO_DIR = os.environ.get("SE_REPO_DIR", "/workspace/self_explainer")
if not os.path.isdir(os.path.join(REPO_DIR, "se")):
    REPO_DIR = os.path.abspath(".." if os.path.isdir("../se") else ".")
sys.path.insert(0, os.path.join(REPO_DIR, "se"))

import se_env

se_env.ensure_deps()


In [ ]:
# --- environment ------------------------------------------------------------
# Caches and outputs go on the pod's volume, never the container disk: /workspace is what
# survives a stopped or terminated pod, and the 8B checkpoint alone is 16 GB. HF_TOKEN comes
# from the pod template's environment or <volume>/.hf_token — there is no prompt to answer,
# because a preempted pod restarts with nobody watching.
#
# The v -> Qv transform is CPU work; only the live-vs-offline spot check loads the
# 8B target.
env = se_env.bootstrap(repo_dir=REPO_DIR, gpu="optional", min_vram_gb=24)

import se_config as C


In [ ]:
import json

import torch

import rotate as R
import se_common as S

tokenizer = S.load_tokenizer()
Q, q_seed = R.load_rotation(f"{C.ROTATION_DIR}/Q_seed{C.Q_SEED}.pt")
print(f"loaded Q (seed {q_seed}), shape {tuple(Q.shape)}")

gate = json.load(open(f"{C.REPORTS_DIR}/invariance_gate.json"))
assert gate["fold_and_rotate"]["passed"], "NB01's invariance gate did not pass. Stop."
print("NB01 gate: PASSED")


## 1. Build the shared base dataset

Shuffle with `SEED`, take the first 10k rows, render prompts. 
Template choice consumes a `random.Random(SEED)` in dataset order, exactly as the base repo does it, so a given row gets the same template it would have there.


In [ ]:
base = S.build_act_dataset(tokenizer, seed=C.SEED, prefix=C.ACT_DATASET_PREFIX)
print(f"{len(base)} examples")
print(f"fields: {base.column_names}")
print()
print("example user turn:")
print(" ", base[0]["messages"][0]["content"][:300])
print("example target:")
print(" ", base[0]["messages"][1]["content"][:200])
print(f"chunk_id distribution: "
      f"{ {c: base['chunk_id'].count(c) for c in sorted(set(base['chunk_id']))} }")


## 2. Emit one dataset per rotation arm


In [ ]:
import time

ARMS = {
    "identity": None,
    "Q": Q,
    # optional third arm (§7.1) — see the note below before relying on it
    "Qscaled": R.random_orthogonal_scaled(Q.shape[0], seed=C.Q_SEED, log_scale=1.0),
}

BUILD_ARMS = ["identity", "Q"]     # add "Qscaled" if the budget allows

paths = {}
for arm in BUILD_ARMS:
    M = ARMS[arm]
    t0 = time.time()
    ds = base if M is None else S.transform_vectors(base, M)
    paths[arm] = S.save_ready_dataset(ds, arm)
    print(f"{arm:>10}: {len(ds)} rows -> {paths[arm]}  ({time.time() - t0:.0f}s)")


### On the `Qscaled` arm

We also have the `Q ∘ diag(s)` arm. 
We make this because `Q` preserves norms, angles, anisotropy, effective rank, adn the whole covariance spectrum.
So, we want to also destroy the fact that they have the same distributional shape.
Including a diagonal scaling means that we no longer compute `M`'s function, but we run it to distort the activation distribution which the explainer is given.
The labels stay attached to the original `M`, so we measure the extent to which the explainers advantage depends on the input distribution matching its own.


## 3. Verify: only the vectors moved

The budget rests on labels being reusable, so we check this.


In [ ]:
from datasets import load_from_disk

ident = load_from_disk(S.ready_dataset_path("identity"))
rot = load_from_disk(S.ready_dataset_path("Q"))

assert len(ident) == len(rot)
n_probe = 500

same_messages = all(ident[i]["messages"] == rot[i]["messages"] for i in range(n_probe))
same_labels = all(ident[i]["is_different"] == rot[i]["is_different"] for i in range(n_probe))
same_chunks = all(ident[i]["chunk_id"] == rot[i]["chunk_id"] for i in range(n_probe))
same_conts = all(ident[i]["ablated_continuation"] == rot[i]["ablated_continuation"]
                 for i in range(n_probe))

print(f"prompts identical      : {same_messages}")
print(f"has-changed identical  : {same_labels}")
print(f"continuations identical: {same_conts}")
print(f"chunk ids identical    : {same_chunks}")
assert same_messages and same_labels and same_chunks and same_conts

vi = torch.tensor([ident[i]["patch_position"]["intervention_vector"] for i in range(n_probe)],
                  dtype=torch.float64)
vr = torch.tensor([rot[i]["patch_position"]["intervention_vector"] for i in range(n_probe)],
                  dtype=torch.float64)

print(f"\nvectors differ         : {not torch.allclose(vi, vr)}")
print(f"rotation is exact      : max |Qv - v_rot| = {((vi @ Q.T) - vr).abs().max().item():.2e}")
print(f"norms preserved        : max |‖Qv‖-‖v‖| = "
      f"{(vi.norm(dim=1) - vr.norm(dim=1)).abs().max().item():.2e}")


## 4. Live vs offline: what licenses the shortcut

We need to ensure that `Qv` is what `M_Q` actually produces.
We compute the tensor in two ways: multiply the cached vector by `Q` and alternatively run `M_Q` and read the activations, and check that they agree.


In [ ]:
RUN_LIVE_CHECK = env["has_gpu"]       # the one step here that needs the 8B target

if RUN_LIVE_CHECK:
    from transformers import AutoModelForCausalLM

    import rotate as R

    tok = tokenizer
    m = AutoModelForCausalLM.from_pretrained(
        C.TARGET_MODEL_ID, dtype=torch.bfloat16, device_map="auto").eval()
    R.fold_rmsnorm_gains(m)
    R.apply_rotation(m, Q)

    N_SPOT = 32
    errs = []
    for i in range(N_SPOT):
        ex = ident[i]
        ids = torch.tensor([tok.convert_tokens_to_ids(ex["input_tokens"])], device=m.device)
        with torch.no_grad():
            out = m(ids, output_hidden_states=True)
        # rebuild the cached vector the way the dataset did: mean over the layer chunk at
        # the patched position. Field names come from NB00's schema dump.
        layers = ex["layer"]
        pos = ex["patch_position"].get("position", ex["patch_position"].get("index"))
        if pos is None:
            print("patch position field not resolved — see NB01's POSITION_KEYS")
            break
        live = torch.stack([out.hidden_states[l + 1][0, pos] for l in layers]).float().mean(0)
        cached = torch.tensor(rot[i]["patch_position"]["intervention_vector"])
        errs.append(((live.cpu() - cached).norm() / cached.norm()).item())

    if errs:
        import numpy as np
        print(f"relative error, live M_Q vs offline Qv, over {len(errs)} examples:")
        print(f"  median {np.median(errs):.4f}   max {max(errs):.4f}")
        print("\nbf16 activations carry ~3 decimal digits, so errors of a few times 1e-3")
        print("are the arithmetic, not the transform. Percent-level errors are not.")

    del m
    import gc
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("skipped — no GPU visible on this pod. This check is what licenses the offline\n"
          "shortcut, so run it on a GPU pod before trusting the rotated dataset.")
    print("v2 §6 lists this as an exit criterion for Phase 2; do not skip it permanently")


## 5. What the rotation did to coordinate alignment

The paper's section 3.4 and Table 4 correlate explainer quality with activation alignment. 
The rotation is supposed to send that alignment to chance. 
We do measure this using cached vectors: how much of a target activation lies along the corresponding coordinate before and after rotation.

The full paired-activations version is done in NB05.

In [ ]:
import numpy as np

vi_n = vi / vi.norm(dim=1, keepdim=True)
vr_n = vr / vr.norm(dim=1, keepdim=True)

# self-correspondence: how much each rotated vector retains of its original direction
cos_self = (vi_n * vr_n).sum(1)
# a null: cosine against an unrelated cached vector
perm = torch.randperm(len(vi_n))
cos_null = (vi_n * vi_n[perm]).sum(1)

print(f"cos(v, Qv)          mean {cos_self.mean():+.4f}  sd {cos_self.std():.4f}")
print(f"cos(v, v')  [null]  mean {cos_null.mean():+.4f}  sd {cos_null.std():.4f}")
print(f"\nexpected null scale for d={vi.shape[1]}: ~{1/np.sqrt(vi.shape[1]):.4f}")
print("\nRotation sends coordinate correspondence to the null scale, which is the point:")
print("every scalar the explainer could read off a fixed coordinate is now scrambled,")
print("while the model computing those activations is unchanged.")


## 6. Carry forward

`act_ready_identity` and `act_ready_Q` are the training inputs for NB03–NB05. Same rows, same
order, same labels; different coordinates.

Next: **NB03** runs the core sweep, Cfull × {identity, Q}.


In [ ]:
for arm, p in paths.items():
    print(f"{arm:>10}: {p}")
print(f"\nbase rows: {len(base)}   eval held out: last {C.EVAL_SIZE}")
